# Phase 1: AutoPET-III (PSMA-PET-CT-Lesions) -- Extract Lesion Features

Processes AutoPET-III DICOM data from Google Drive. Adapted from `process_autopet_i.ipynb` with three key differences:

1. **SUV from raw DICOM** -- AutoPET-I shipped pre-computed SUV NIfTI from FDAT; AutoPET-III is raw DICOM. SUV is computed via the validated `src/preprocess/suv_conversion.py` (pre-reg §3.3, 9/9 pass on three scanner models, max_rel_diff 0.00000%).
2. **Heterogeneous storage** -- 1,107 series saved as `{uid}.zip`, 684 as extracted `{uid}/` directories. Loader tries zip first, falls back to directory.
3. **No SEG masks** -- AutoPET-III TCIA is imaging-only. Segmentation must come from nnU-Net v2 inference (separate notebook, blocked on checkpoint download). This notebook expects SEG NIfTI at `$WORK_DIR/autopet_iii/segmentations/{series_uid}.nii.gz` and fails gracefully (skip + log) if missing.

Includes SUVpeak in the same Step 4 (no need for a separate Step 9 augmentation).

**Pre-registration:** https://doi.org/10.17605/OSF.IO/4KAZN  
**Schema target:** identical to AutoPET-I `lesion_features.parquet` -- `case_id`, `lesion_id`, `suvmax`, `suvmean`, `suvpeak`, `tlg`, `volume_ml`, `n_voxels`, `centroid_*`, `voxel_spacing_*`, plus `series_uid`, `radionuclide`; `dataset='autopet_iii'`; `tracer in {'FDG','PSMA'}`; `vendor in {'Siemens','GE'}`.

In [ ]:
# Working-directory configuration:
# Set the WORK_DIR environment variable (or override here) to point at
# the local or networked folder that holds the raw cohort data.
# Default Colab convention: the mounted Drive root.
import os
WORK_DIR = os.environ.get('WORK_DIR', '/content/drive/MyDrive')
print(f'WORK_DIR = {WORK_DIR}')


In [ ]:
# Step 1: Mount Drive and install deps
from google.colab import drive
drive.mount('/content/drive')

!pip install -q pydicom SimpleITK nibabel tcia_utils

In [ ]:
# Step 2: Upload suv_conversion.py from project source
# When the file picker opens, select:
#   src/preprocess/suv_conversion.py
# This is the validated module (pre-reg §3.3, 9/9 pass).
# Any prior copy in /content is removed first to avoid Colab's auto-rename behaviour.
import os
import sys
from google.colab import files

target = '/content/suv_conversion.py'
if os.path.exists(target):
    os.remove(target)
    print(f'Removed stale {target}')

uploaded = files.upload()
assert 'suv_conversion.py' in uploaded, 're-run and select suv_conversion.py'

sys.path.insert(0, '/content')
sys.modules.pop('suv_conversion', None)  # drop any cached old version

from suv_conversion import extract_pet_metadata, dicom_series_to_suv_sitk
print('Imported extract_pet_metadata, dicom_series_to_suv_sitk')

In [ ]:
# Step 3: Inspect Drive layout and build PT-series manifest
# Two storage formats coexist on Drive:
#   {uid}.zip                     -- 1,107 series (post-2026-04-26 refactor)
#   {uid}/<DICOM files>           -- 684 series (pre-refactor extracted dirs)
# We filter to Modality=='PT' via TCIA's getSeries (the same call the download notebook used).
import zipfile
import pandas as pd
from tcia_utils import nbia

DRIVE_ROOT = f'{WORK_DIR}/autopet_iii'

# Pull the full series manifest from TCIA
series_df = nbia.getSeries(collection='PSMA-PET-CT-Lesions', format='df')
print(f'Total series in collection: {len(series_df)}')
if 'Modality' in series_df.columns:
    print(f'Modalities: {series_df["Modality"].value_counts().to_dict()}')
    pt_series = series_df[series_df['Modality'] == 'PT'].copy()
else:
    raise RuntimeError('Modality column missing from TCIA response')
print(f'PT series: {len(pt_series)}')

# Discover what's actually on Drive
drive_entries = set(os.listdir(DRIVE_ROOT))
zips_on_drive = {e[:-4] for e in drive_entries if e.endswith('.zip')}
dirs_on_drive = {e for e in drive_entries
                 if not e.endswith('.zip')
                 and not e.startswith('_')
                 and not e.startswith('.')
                 and os.path.isdir(os.path.join(DRIVE_ROOT, e))}
print(f'Drive: {len(zips_on_drive)} zips + {len(dirs_on_drive)} extracted dirs = {len(zips_on_drive) + len(dirs_on_drive)} series total')

# Build the working manifest: PT series available locally
pt_uids = set(pt_series['SeriesInstanceUID'])
pt_zips = pt_uids & zips_on_drive
pt_dirs = pt_uids & dirs_on_drive
pt_missing = pt_uids - zips_on_drive - dirs_on_drive
print(f'\nPT series accounted for: {len(pt_zips) + len(pt_dirs)} / {len(pt_uids)}')
print(f'  as zip:       {len(pt_zips)}')
print(f'  as directory: {len(pt_dirs)}')
if pt_missing:
    print(f'  MISSING:      {len(pt_missing)} (re-run download notebook for these)')

# Manifest as a DataFrame for the rest of the notebook
manifest = pt_series[pt_series['SeriesInstanceUID'].isin(pt_zips | pt_dirs)].copy()
manifest['storage'] = manifest['SeriesInstanceUID'].apply(
    lambda u: 'zip' if u in pt_zips else 'dir'
)
print(f'\nWorking manifest: {len(manifest)} PT series')

In [ ]:
# Step 4: Loader helper -- try-zip-first / fall-back-to-dir
# Returns (path_to_dicom_dir, cleanup_callable). For zip storage we extract
# to /tmp; for dir storage we point at Drive directly. Cleanup is a no-op
# for dir storage so we don't accidentally delete Drive data.
import shutil
import tempfile

def load_pet_series(series_uid, drive_root=DRIVE_ROOT):
    zip_path = os.path.join(drive_root, f'{series_uid}.zip')
    dir_path = os.path.join(drive_root, series_uid)
    if os.path.exists(zip_path):
        tmp_dir = tempfile.mkdtemp(prefix='petseries_')
        with zipfile.ZipFile(zip_path, 'r') as zf:
            zf.extractall(tmp_dir)
        # Some TCIA zips contain a single inner directory; flatten if so
        entries = os.listdir(tmp_dir)
        if len(entries) == 1 and os.path.isdir(os.path.join(tmp_dir, entries[0])):
            inner = os.path.join(tmp_dir, entries[0])
        else:
            inner = tmp_dir
        return inner, lambda: shutil.rmtree(tmp_dir, ignore_errors=True)
    elif os.path.isdir(dir_path):
        return dir_path, lambda: None
    else:
        raise FileNotFoundError(f'No zip or directory for {series_uid} under {drive_root}')

# Smoke test on one series of each storage type
import glob
for storage in ('zip', 'dir'):
    sub = manifest[manifest['storage'] == storage]
    if len(sub) == 0:
        continue
    test_uid = sub.iloc[0]['SeriesInstanceUID']
    print(f'\nTesting {storage} loader on {test_uid[:32]}...')
    dcm_dir, cleanup = load_pet_series(test_uid)
    n_dcm = len(glob.glob(os.path.join(dcm_dir, '*.dcm')))
    if n_dcm == 0:
        # Some zips ship without .dcm extension; count any files instead
        n_dcm = sum(1 for _ in os.scandir(dcm_dir) if _.is_file())
    print(f'  loader OK: {n_dcm} files at {dcm_dir}')
    cleanup()

In [ ]:
# Step 5: Single-series end-to-end test -- DICOM -> SUV -> sanity
# Confirms the validated SUV pipeline produces typical-range values on a real
# AutoPET-III series before we run it on all 597. If SUV ranges look wrong
# here, STOP -- do not proceed to Step 7.
import numpy as np
import SimpleITK as sitk

test_uid = manifest.iloc[0]['SeriesInstanceUID']
print(f'Test series: {test_uid}')

dcm_dir, cleanup = load_pet_series(test_uid)
try:
    # Pick any DICOM file in the series for metadata
    any_dcm = next(iter(
        glob.glob(os.path.join(dcm_dir, '*.dcm')) or
        [os.path.join(dcm_dir, e) for e in os.listdir(dcm_dir)
         if os.path.isfile(os.path.join(dcm_dir, e))]
    ))
    meta = extract_pet_metadata(any_dcm)
    print(f'\nMetadata:')
    print(f'  patient_id:        {meta.patient_id}')
    print(f'  manufacturer:      {meta.manufacturer}')
    print(f'  model:             {meta.manufacturer_model}')
    print(f'  radionuclide:      {meta.radionuclide}')
    print(f'  weight:            {meta.patient_weight_kg} kg')
    print(f'  injected dose:     {meta.injected_dose_bq:.2e} Bq')
    print(f'  uptake time:       {meta.uptake_time_sec/60:.1f} min')
    print(f'  decay factor:      {meta.decay_factor:.4f}')

    suv_image = dicom_series_to_suv_sitk(dcm_dir, meta)
    suv = sitk.GetArrayFromImage(suv_image)
    spacing = suv_image.GetSpacing()  # (x,y,z) in mm

    print(f'\nSUV volume:')
    print(f'  shape:             {suv.shape}')
    print(f'  spacing (mm):      ({spacing[0]:.2f}, {spacing[1]:.2f}, {spacing[2]:.2f})')
    print(f'  SUV range:         [{suv.min():.2f}, {suv.max():.2f}]')
    print(f'  SUV mean (>0):     {suv[suv > 0].mean():.2f}')
    print(f'  voxels > 0:        {(suv > 0).sum():,}')

    # Sanity gates
    if suv.min() < -0.1:
        print(f'\nWARNING: negative SUV values present (min={suv.min():.3f}) -- check pipeline')
    if suv.max() > 200:
        print(f'\nWARNING: SUV max > 200 ({suv.max():.1f}) -- almost certainly artifact, not biology')
    if 0 <= suv.min() <= 0.01 and 1 < suv.max() < 200:
        print(f'\nSUV ranges look correct -- safe to proceed.')
finally:
    cleanup()

## Step 6: nnU-Net v2 segmentation inference

**See sibling notebook: `kaggle_notebooks/segment_autopet_iii.ipynb`.**

That notebook handles the full segmentation pipeline:

- Downloads the LesionTracer checkpoint (Zenodo [14007247](https://zenodo.org/records/14007247), 3.8 GB, MD5 `566016409b0bd14770c0b57c1f2873f1`) to Drive
- Installs the LesionTracer-forked nnU-Net v2 ([MIC-DKFZ/autopet-3-submission](https://github.com/MIC-DKFZ/autopet-3-submission)) — required because the checkpoint depends on a custom trainer class not in vanilla `nnunetv2`
- Pairs each PT series with its matching CT (shared `StudyInstanceUID`)
- Builds two-channel input per case: `_0000.nii.gz` = CT (HU, resampled to PET grid) and `_0001.nii.gz` = SUV PET
- Runs `nnUNetv2_predict_from_modelfolder`, writing binary lesion masks (0=bg, 1=lesion) directly to:
  ```
  $WORK_DIR/autopet_iii/segmentations/{series_uid}.nii.gz
  ```
  — exactly the path Step 7 below expects.

**Step 7 below is fail-soft on missing SEGs**, so you can run the two notebooks in any interleaved order — re-running Step 7 after each segmentation batch will pick up the newly-completed series.

**Compute estimate:** ~5-7 GPU-hours (single fold) or ~25-35 GPU-hours (5-fold ensemble) for all 597 PT series on a T4.

In [ ]:
# Step 7: Lesion feature extraction (DICOM SUV + nnU-Net SEG -> parquet)
# Includes SUVpeak in the same pass (no Step 9 augmentation needed for AutoPET-III).
# Schema matches AutoPET-I lesion_features.parquet plus series_uid + radionuclide.
# Resumable via _autopet_iii_processed.txt. Fail-soft on missing SEG.

import time
import numpy as np
import nibabel as nib
import SimpleITK as sitk
from scipy import ndimage

MIN_VOLUME_ML = 1.0  # pre-reg §3.2
SPHERE_VOLUME_ML = 1.0  # pre-reg §3.3 (PERCIST/QIBA)

SEG_DIR = os.path.join(DRIVE_ROOT, 'segmentations')
OUTPUT_PATH = os.path.join(DRIVE_ROOT, 'autopet_iii_lesions.parquet')
PROGRESS_PATH = os.path.join(DRIVE_ROOT, '_autopet_iii_processed.txt')
ERROR_PATH = os.path.join(DRIVE_ROOT, '_autopet_iii_errors.txt')

os.makedirs(SEG_DIR, exist_ok=True)  # ensures path exists for fail-soft check

# --- helpers ---
def vendor_from_manufacturer(mfr):
    s = (mfr or '').upper()
    if 'SIEMENS' in s:
        return 'Siemens'
    if 'GE' in s:
        return 'GE'
    if 'PHILIPS' in s:
        return 'Philips'
    return 'Unknown'

def tracer_from_radionuclide(r):
    """DEPRECATED for AutoPET-III. The PSMA-PET-CT-Lesions collection contains
    BOTH 18F-PSMA and 68Ga-PSMA tracers (369 18F + 228 68Ga = 597 total per
    clinical_metadata.tsv); patient ID prefix `PSMA_` confirms all are PSMA-
    target imaging. Step 7 now hardcodes tracer='PSMA' for AutoPET-III.
    Function retained for any future cohort that mixes FDG + PSMA per-series.
    Per-radionuclide sensitivity analyses use the `radionuclide` column directly."""
    if r is None:
        return 'Unknown'
    s = str(r).upper()
    if 'F-18' in s or '18F' in s or 'FLUORINE' in s:
        return 'FDG'
    if 'GA-68' in s or '68GA' in s or 'GALLIUM' in s:
        return 'PSMA'
    return 'Unknown'

def compute_suvpeak(suv, spacing_zyx, max_voxel_idx, sphere_volume_ml=1.0):
    radius_mm = (3.0 * sphere_volume_ml * 1000.0 / (4.0 * np.pi)) ** (1.0/3.0)
    spacing_arr = np.array(spacing_zyx, dtype=np.float64)
    half_vox = np.ceil(radius_mm / spacing_arr).astype(int)
    cz, cy, cx = max_voxel_idx
    z0, z1 = max(cz - half_vox[0], 0), min(cz + half_vox[0] + 1, suv.shape[0])
    y0, y1 = max(cy - half_vox[1], 0), min(cy + half_vox[1] + 1, suv.shape[1])
    x0, x1 = max(cx - half_vox[2], 0), min(cx + half_vox[2] + 1, suv.shape[2])
    zz, yy, xx = np.meshgrid(
        np.arange(z0, z1) - cz,
        np.arange(y0, y1) - cy,
        np.arange(x0, x1) - cx,
        indexing='ij',
    )
    dist_mm = np.sqrt((zz * spacing_arr[0])**2
                      + (yy * spacing_arr[1])**2
                      + (xx * spacing_arr[2])**2)
    sphere_mask = dist_mm <= radius_mm
    return float(suv[z0:z1, y0:y1, x0:x1][sphere_mask].mean())

def compute_softmax_features(npz_path, lesion_mask):
    """Load nnU-Net softmax probabilities and compute mean + entropy for a lesion.
    Returns (softmax_mean, softmax_entropy) or (None, None) if .npz unavailable
    or shape-mismatched. Pre-reg §4.2 CQR predictors."""
    if not os.path.exists(npz_path):
        return None, None
    try:
        data = np.load(npz_path)
        key = 'probabilities' if 'probabilities' in data.files else data.files[0]
        probs = data[key]
        # nnU-Net saves (n_classes, *spatial). Take tumour (foreground) channel.
        if probs.ndim == 4 and probs.shape[0] == 2:
            p = probs[1]
        elif probs.ndim == 3:
            p = probs
        else:
            return None, None
        if p.shape != lesion_mask.shape:
            # try transpose -- nnU-Net may save in (z,y,x) vs nibabel (x,y,z)
            if p.T.shape == lesion_mask.shape:
                p = p.T
            else:
                return None, None
        p_lesion = p[lesion_mask > 0].astype(np.float32)
        if len(p_lesion) == 0:
            return float('nan'), float('nan')
        softmax_mean = float(p_lesion.mean())
        # Voxel-wise binary Shannon entropy, averaged over lesion voxels
        eps = np.float32(1e-6)
        p_safe = np.clip(p_lesion, eps, 1 - eps)
        ent = -p_safe * np.log(p_safe) - (1 - p_safe) * np.log(1 - p_safe)
        return softmax_mean, float(ent.mean())
    except Exception:
        return None, None

# --- resume state ---
processed = set()
if os.path.exists(PROGRESS_PATH):
    with open(PROGRESS_PATH) as f:
        processed = set(line.strip() for line in f if line.strip())
    print(f'Resuming: {len(processed)} series already processed')

all_features = []
if os.path.exists(OUTPUT_PATH) and processed:
    existing = pd.read_parquet(OUTPUT_PATH)
    all_features = existing.to_dict('records')
    print(f'Loaded {len(all_features)} existing lesion features')

# --- iterate ---
to_process = [u for u in manifest['SeriesInstanceUID'] if u not in processed]
print(f'PT series to process: {len(to_process)} / {len(manifest)}')
print(f'Looking for SEGs at: {SEG_DIR}/{{series_uid}}.nii.gz\n')

start = time.time()
n_done = 0
n_skipped_no_seg = 0
n_skipped_shape = 0
errors = []

for series_uid in to_process:
    seg_path = os.path.join(SEG_DIR, f'{series_uid}.nii.gz')
    if not os.path.exists(seg_path):
        n_skipped_no_seg += 1
        continue  # do NOT mark processed -- will retry once SEG arrives

    cleanup = lambda: None
    try:
        dcm_dir, cleanup = load_pet_series(series_uid)
        any_dcm = next(iter(
            glob.glob(os.path.join(dcm_dir, '*.dcm')) or
            [os.path.join(dcm_dir, e) for e in os.listdir(dcm_dir)
             if os.path.isfile(os.path.join(dcm_dir, e))]
        ))
        meta = extract_pet_metadata(any_dcm)
        suv_image = dicom_series_to_suv_sitk(dcm_dir, meta)
        suv = sitk.GetArrayFromImage(suv_image).astype(np.float64)  # (z,y,x)
        spacing_xyz = suv_image.GetSpacing()  # (x,y,z) per SimpleITK
        spacing_zyx = (spacing_xyz[2], spacing_xyz[1], spacing_xyz[0])

        seg = nib.load(seg_path).get_fdata().astype(np.int32)
        npz_path = seg_path.replace('.nii.gz', '.npz')
        # nibabel convention is (x,y,z); SimpleITK array is (z,y,x). Transpose if needed.
        if seg.shape != suv.shape:
            if seg.T.shape == suv.shape:
                seg = seg.T
            else:
                errors.append((series_uid, f'shape mismatch SUV {suv.shape} SEG {seg.shape}'))
                n_skipped_shape += 1
                cleanup()
                continue

        binary = (seg > 0).astype(np.int32)
        labelled, n_comp = ndimage.label(binary)
        voxel_vol_ml = float(np.prod(spacing_zyx)) / 1000.0

        case_id = meta.patient_id
        vendor = vendor_from_manufacturer(meta.manufacturer)
        # AutoPET-III is PSMA-target imaging (both 18F-PSMA and 68Ga-PSMA tracers); see clinical_metadata.tsv
        tracer = 'PSMA'

        for comp_id in range(1, n_comp + 1):
            comp_mask = labelled == comp_id
            n_vox = int(comp_mask.sum())
            vol_ml = n_vox * voxel_vol_ml
            if vol_ml < MIN_VOLUME_ML:
                continue

            suv_lesion = suv[comp_mask]
            coords = np.argwhere(comp_mask)
            centroid = coords.mean(axis=0)

            suv_in_lesion = np.where(comp_mask, suv, -np.inf)
            max_idx = np.unravel_index(np.argmax(suv_in_lesion), suv.shape)
            suvpeak = compute_suvpeak(suv, spacing_zyx, max_idx, SPHERE_VOLUME_ML)

            sm_mean, sm_entropy = compute_softmax_features(npz_path, comp_mask)

            all_features.append({
                'case_id': case_id,
                'lesion_id': comp_id,
                'series_uid': series_uid,
                'suvmax': float(suv_lesion.max()),
                'suvmean': float(suv_lesion.mean()),
                'suvpeak': float(suvpeak),
                'tlg': float(suv_lesion.mean()) * vol_ml,
                'volume_ml': vol_ml,
                'n_voxels': n_vox,
                'centroid_0': float(centroid[0]),
                'centroid_1': float(centroid[1]),
                'centroid_2': float(centroid[2]),
                'voxel_spacing_0': float(spacing_zyx[0]),
                'voxel_spacing_1': float(spacing_zyx[1]),
                'voxel_spacing_2': float(spacing_zyx[2]),
                'dataset': 'autopet_iii',
                'tracer': tracer,
                'vendor': vendor,
                'radionuclide': meta.radionuclide,
                'softmax_mean': sm_mean,
                'softmax_entropy': sm_entropy,
            })

        cleanup()
        with open(PROGRESS_PATH, 'a') as f:
            f.write(series_uid + '\n')
        processed.add(series_uid)
        n_done += 1

    except Exception as e:
        errors.append((series_uid, str(e)))
        cleanup()

    if n_done > 0 and n_done % 25 == 0:
        elapsed = time.time() - start
        rate = n_done / elapsed * 3600
        rem = (len(to_process) - n_done - n_skipped_no_seg) / rate if rate > 0 else 0
        print(f'  {len(processed)}/{len(manifest)} processed | '
              f'{len(all_features)} lesions | '
              f'{n_skipped_no_seg} skipped (no SEG) | '
              f'{len(errors)} errors | '
              f'{rate:.0f} series/hr | ~{rem:.1f}h remaining')
        pd.DataFrame(all_features).to_parquet(OUTPUT_PATH, index=False)

# Final save
if all_features:
    pd.DataFrame(all_features).to_parquet(OUTPUT_PATH, index=False)
elapsed = time.time() - start
print(f'\nDone in {elapsed/3600:.2f}h')
print(f'  Lesions extracted:  {len(all_features)}')
print(f'  Series processed:   {len(processed)} / {len(manifest)}')
print(f'  Skipped (no SEG):   {n_skipped_no_seg}  -- will retry on rerun once segmentations land')
print(f'  Skipped (shape):    {n_skipped_shape}')
print(f'  Errors:             {len(errors)}')
if errors:
    with open(ERROR_PATH, 'a') as f:
        for u, e in errors:
            f.write(f'{u}\t{e}\n')
    print(f'  Error log:          {ERROR_PATH}')
    print(f'  First 5: ')
    for u, e in errors[:5]:
        print(f'    {u[:32]}...: {e[:120]}')

In [ ]:
# Step 8: Summary stats (mirrors process_autopet_i.ipynb Step 5)
OUTPUT_PATH = os.path.join(DRIVE_ROOT, 'autopet_iii_lesions.parquet')

if os.path.exists(OUTPUT_PATH):
    df = pd.read_parquet(OUTPUT_PATH)
    print(f'Total lesions (>=1 mL): {len(df)}')
    print(f'Total cases (patients): {df["case_id"].nunique()}')
    print(f'Total PT series:        {df["series_uid"].nunique()}')
    print(f'\nTracer breakdown:')
    print(df.groupby('tracer').agg(
        n_lesions=('lesion_id', 'count'),
        n_cases=('case_id', 'nunique'),
        median_suvmax=('suvmax', 'median'),
    ).to_string())
    print(f'\nVendor breakdown:')
    print(df.groupby('vendor').agg(
        n_lesions=('lesion_id', 'count'),
        n_cases=('case_id', 'nunique'),
    ).to_string())
    print(f'\nLesions per case: mean={df.groupby("case_id").size().mean():.1f}, '
          f'median={df.groupby("case_id").size().median():.0f}, '
          f'max={df.groupby("case_id").size().max()}')
    print(f'\nVolume (mL):  median={df["volume_ml"].median():.1f}, '
          f'IQR=[{df["volume_ml"].quantile(0.25):.1f}, {df["volume_ml"].quantile(0.75):.1f}], '
          f'range=[{df["volume_ml"].min():.1f}, {df["volume_ml"].max():.1f}]')
    print(f'SUVmax:       median={df["suvmax"].median():.1f}, '
          f'IQR=[{df["suvmax"].quantile(0.25):.1f}, {df["suvmax"].quantile(0.75):.1f}], '
          f'range=[{df["suvmax"].min():.1f}, {df["suvmax"].max():.1f}]')
    print(f'SUVpeak:      median={df["suvpeak"].median():.1f}, '
          f'IQR=[{df["suvpeak"].quantile(0.25):.1f}, {df["suvpeak"].quantile(0.75):.1f}]')
    print(f'SUVmean:      median={df["suvmean"].median():.1f}')
    print(f'TLG:          median={df["tlg"].median():.1f}')
    ratio = df['suvpeak'] / df['suvmax']
    print(f'\nSUVpeak/SUVmax ratio (typical 0.4-0.9): median={ratio.median():.3f}')
    print(f'  ratio < 0.5: {int((ratio < 0.5).sum())} lesions  (single-voxel dominated)')
    print(f'  ratio < 0.3: {int((ratio < 0.3).sum())} lesions  (severe single-voxel artifact)')
    print(f'\nVolume quartile distribution:')
    df['vol_q'] = pd.qcut(df['volume_ml'], 4, labels=['Q1','Q2','Q3','Q4'])
    for q in ['Q1','Q2','Q3','Q4']:
        sub = df[df['vol_q'] == q]
        print(f'  {q}: n={len(sub)}, vol=[{sub["volume_ml"].min():.1f}, {sub["volume_ml"].max():.1f}] mL, '
              f'median SUVmax={sub["suvmax"].median():.1f}')
    print(f'\nFlags:')
    print(f'  SUVmax > 50 (§3.9 trigger): {(df["suvmax"] > 50).sum()} lesions in {df.loc[df["suvmax"] > 50, "case_id"].nunique()} cases')
    print(f'  SUVmax < 0:                {(df["suvmax"] < 0).sum()}')
else:
    print('No feature table yet -- run Step 7 once segmentations are available.')

In [ ]:
# Step 9: Triage SUVmax > 50 outliers per §3.9 (mirrors AutoPET-I Step 7)
# Same triage taxonomy: A_extreme_suv (>150), B_small_high_suv (<2 mL),
# C_focal_hotspot (uniformity <0.30), D_likely_real (auto-retain).
REVIEW_PATH = os.path.join(DRIVE_ROOT, 'autopet_iii_outlier_review.csv')

df = pd.read_parquet(OUTPUT_PATH)
flagged = df[df['suvmax'] > 50].copy()
flagged['centroid_z_mm'] = flagged['centroid_2'] * flagged['voxel_spacing_2']
flagged['suv_uniformity'] = flagged['suvmean'] / flagged['suvmax']

def categorise(row):
    if row['suvmax'] > 150:
        return 'A_extreme_suv'
    if row['volume_ml'] < 2.0:
        return 'B_small_high_suv'
    if row['suv_uniformity'] < 0.30:
        return 'C_focal_hotspot'
    return 'D_likely_real'

flagged['triage_category'] = flagged.apply(categorise, axis=1)
flagged['needs_image_review'] = flagged['triage_category'] != 'D_likely_real'
flagged = flagged.sort_values(['triage_category', 'suvmax'], ascending=[True, False])

out_cols = [
    'case_id', 'series_uid', 'lesion_id', 'tracer', 'vendor',
    'triage_category', 'needs_image_review',
    'suvmax', 'suvpeak', 'suvmean', 'suv_uniformity', 'volume_ml',
    'centroid_2', 'centroid_z_mm', 'voxel_spacing_2',
]
flagged[out_cols].to_csv(REVIEW_PATH, index=False)

print(f'Total SUVmax > 50 lesions flagged: {len(flagged)}')
print(f'Unique cases affected: {flagged["case_id"].nunique()}')
print(f'\nTriage breakdown:')
for cat in sorted(flagged['triage_category'].unique()):
    sub = flagged[flagged['triage_category'] == cat]
    print(f'  {cat}: n={len(sub)}, '
          f'SUVmax median={sub["suvmax"].median():.1f}, '
          f'volume median={sub["volume_ml"].median():.2f} mL, '
          f'SUVpeak/SUVmax median={(sub["suvpeak"]/sub["suvmax"]).median():.3f}')

needs_n = int(flagged['needs_image_review'].sum())
needs_cases = flagged[flagged['needs_image_review']]['case_id'].nunique()
print(f'\nNeeds image review: {needs_n} lesions across {needs_cases} cases')
print(f'Auto-retain (likely real disease): {len(flagged) - needs_n} lesions')
print(f'\nReview CSV: {REVIEW_PATH}')

In [ ]:
# Step 10: Per-case concentration diagnostic (mirrors AutoPET-I Step 8)
# Decides whether §3.9 review can be case-level (concentrated) or must be lesion-level (distributed).
flagged = pd.read_csv(REVIEW_PATH)
df = pd.read_parquet(OUTPUT_PATH)

per_case = flagged.groupby('case_id').agg(
    flagged_n=('lesion_id', 'count'),
    cat_A=('triage_category', lambda s: (s == 'A_extreme_suv').sum()),
    cat_B=('triage_category', lambda s: (s == 'B_small_high_suv').sum()),
    cat_C=('triage_category', lambda s: (s == 'C_focal_hotspot').sum()),
    cat_D=('triage_category', lambda s: (s == 'D_likely_real').sum()),
    suvmax_max=('suvmax', 'max'),
    suvmax_med=('suvmax', 'median'),
).join(df.groupby('case_id').size().rename('total_lesions_in_case'), how='left')
per_case['flagged_pct'] = (per_case['flagged_n'] / per_case['total_lesions_in_case'] * 100).round(1)
per_case = per_case.sort_values('flagged_n', ascending=False)

print('Per-case breakdown of flagged lesions (sorted by flag count):')
print(per_case.to_string())

if (per_case['cat_C'] > 0).any():
    print(f'\nCumulative concentration of Category C:')
    c_only = per_case[per_case['cat_C'] > 0].sort_values('cat_C', ascending=False)
    total_c = int(c_only['cat_C'].sum())
    cum = 0
    for i, (case, row) in enumerate(c_only.iterrows(), 1):
        cum += int(row['cat_C'])
        print(f'  Top {i} ({case}): +{int(row["cat_C"])}  cumulative {cum}/{total_c} ({cum/total_c*100:.0f}%)')
        if cum / total_c >= 0.90:
            print(f'  ...top {i} cases account for >=90% of Category C')
            break
    n_top_50pct = (c_only['cat_C'].cumsum() / total_c >= 0.50).idxmax()
    n_cases_50 = c_only.index.get_loc(n_top_50pct) + 1
    print(f'\nInterpretation:')
    if n_cases_50 <= 3:
        print(f'  Category C is CASE-CONCENTRATED ({n_cases_50} cases hold >=50% of C lesions).')
        print(f'  Mechanism is likely case-level. Review at case level, not lesion level.')
    else:
        print(f'  Category C is DISTRIBUTED ({n_cases_50} cases hold >=50% of C lesions).')
        print(f'  Mechanism likely systematic (necrotic-centred biology, or scanner/protocol effect).')
else:
    print('\nNo Category C lesions -- single-voxel artifact pattern absent in this cohort.')